# Build a model to predict region activity

This notebook shows how to use the ChromBERT-tools subcommand `region_activity_regression` to build a model to predict the region activity.

For the Python API, see [`examples/api/region_activity_regression.ipynb`](../api/region_activity_regression.ipynb).

If you need to use Apptainer container, please refer to the [`apptainer_use.ipynb`](apptainer_use.ipynb) tutorial for detailed instructions on using `apptainer exec` with `chrombert-tools`.

For more details, please refer to the [`region_activity_regression`](https://chrombert-tools.readthedocs.io/en/latest/commands/region_activity_regression.html) command documentation

### 1. Build model

#### Predict the foldchange of region activity between cell-state-transitions

In [ ]:
# Download example data
# Myoblast and fibroblast data: ATAC-seq bigWig and peak files
import subprocess
import os
if not os.path.exists('../data/myoblast_ENCFF647RNC_peak.bed'):
    cmd = f'wget https://www.encodeproject.org/files/ENCFF647RNC/@@download/ENCFF647RNC.bed.gz -O ../data/myoblast_ENCFF647RNC_peak.bed.gz'
    subprocess.run(cmd, shell=True)
    cmd = f"gzip -d ../data/myoblast_ENCFF647RNC_peak.bed.gz"
    subprocess.run(cmd, shell=True)

if not os.path.exists('../data/myoblast_ENCFF149ERN_signal.bigwig'):
    cmd = f'wget https://www.encodeproject.org/files/ENCFF149ERN/@@download/ENCFF149ERN.bigWig -O ../data/myoblast_ENCFF149ERN_signal.bigwig'
    subprocess.run(cmd, shell=True) 


if not os.path.exists('../data/fibroblast_ENCFF184KAM_peak.bed'):
    cmd = f'wget https://www.encodeproject.org/files/ENCFF184KAM/@@download/ENCFF184KAM.bed.gz -O ../data/fibroblast_ENCFF184KAM_peak.bed.gz'
    subprocess.run(cmd, shell=True)
    cmd = f"gzip -d ../data/fibroblast_ENCFF184KAM_peak.bed.gz"
    subprocess.run(cmd, shell=True)


if not os.path.exists('../data/fibroblast_ENCFF361BTT_signal.bigwig'):
    cmd = 'wget https://www.encodeproject.org/files/ENCFF361BTT/@@download/ENCFF361BTT.bigWig -O ../data/fibroblast_ENCFF361BTT_signal.bigwig'
    subprocess.run(cmd, shell=True)

In [2]:
%%bash
# --acc-peak1: the peak file of cell-state 1
# --acc-peak2: the peak file of cell-state 2
# --acc-signal1: the signal file of cell-state 1
# --acc-signal2: the signal file of cell-state 2
# --genome: the genome
# --resolution: the resolution
# --odir: the output directory
# --include-tss-background: include the background regions from TSS, we use the same background regions for both cell-states

# Set the GPU device with CUDA_VISIBLE_DEVICES
export CUDA_VISIBLE_DEVICES=1
chrombert-tools region_activity_regression \
    --acc-peak1 "../data/fibroblast_ENCFF184KAM_peak.bed" \
    --acc-peak2 "../data/myoblast_ENCFF647RNC_peak.bed" \
    --acc-signal1 "../data/fibroblast_ENCFF361BTT_signal.bigwig" \
    --acc-signal2 "../data/myoblast_ENCFF149ERN_signal.bigwig" \
    --genome "hg38" \
    --resolution "1kb" \
    --odir "output_region_activity_regression" \
    --include-tss-background

Stage 1: prepare chromatin accessibility dataset
Processing stage 1: prepare chromatin accessibility dataset
  Mode: two states (fold-change label)
  TSS ± flank background regions: enabled, flank distance: 10000 bp
[state1_0] Region summary - total: 287892, overlapping with ChromBERT: 284160 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 5967
  [state1] merged 1 peak file(s) → 250143 unique ChromBERT regions (after overlap + dedup)
[state2_0] Region summary - total: 373422, overlapping with ChromBERT: 368260 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 7920
  [state2] merged 1 peak file(s) → 324690 unique ChromBERT regions (after overlap + dedup)
[tss_bg] Region summary - total: 19260, overlapping with ChromBERT: 333463 (one region may overlap multiple ChromBERT regions,

/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/lib/python3.9/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [1]
Loading `train_dataloader` to estimate number of stepping batches.
/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/l

Epoch 0:  20%|██        | 800/4000 [02:17<09:09,  5.82it/s, v_num=0]       
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0:   0%|          | 0/250 [00:00<?, ?it/s]


Metric default_validation/pcc improved. New best score: 0.128


Epoch 0:  40%|████      | 1600/4000 [05:00<07:30,  5.33it/s, v_num=0, default_validation/r2=0.00306, default_validation/pcc=0.128, default_validation/scc=0.116, default_validation/mae=0.755, default_validation/mse=1.310, default_validation/rmse=1.150, default_validation/mean=0.0393, default_validation/median=0.0415]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 0:  60%|██████    | 2400/4000 [07:42<05:08,  5.19it/s, v_num=0, default_validation/r2=0.008, default_validation/pcc=0.105, default_validation/scc=0.164, default_validation/mae=0.714, default_validation/mse=1.180, default_validation/rmse=1.090, default_validation/mean=0.128, default_validation/median=0.133]    
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 0:  80%|████████  | 3200/4000 [10:24<02:36,  5.13it/s, v_num=0, default_validation/r2=0.00994, default_validation/pcc=0.106, default_validation/scc=0.175, default_validation/

...

Epoch 3:  60%|██████    | 2400/4000 [08:06<05:24,  4.94it/s, v_num=0, default_validation/r2=0.816, default_validation/pcc=0.905, default_validation/scc=0.823, default_validation/mae=0.344, default_validation/mse=0.234, default_validation/rmse=0.484, default_validation/mean=0.116, default_validation/median=0.0728]
Evaluating the finetuned model performance
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_region_activity_regression/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=3-step=214.ckpt
Loading from pl module, remove prefix 'model.'
Loading from pl module, replace 'pretrain_model' with 'pretrain_model.chrombert'
Loaded 111/111 parameters


/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/lib/python3.9/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


ft_ckpt: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_region_activity_regression/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=3-step=214.ckpt, test_metrics: {'pearsonr': 0.8869209885597229, 'spearmanr': 0.8276695609092712, 'mse': 0.25876349210739136, 'mae': 0.3616968095302582, 'r2': 0.7844354755596655}
Attempt metrics: pearsonr=0.8869209885597229
Accepted run (pearsonr=0.8869 >= 0.2).

Finished stage 2: obtained a fine-tuned ChromBERT
Best pearsonr=0.8869209885597229, metrics={'pearsonr': 0.8869209885597229, 'spearmanr': 0.8276695609092712, 'mse': 0.25876349210739136, 'mae': 0.3616968095302582, 'r2': 0.7844354755596655, 'ft_ckpt': '/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_region_activity_regression/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=3-step=214.ckpt'}
Finished stage 2 (trained)
Stage 3: Predi

Predicting: 100%|██████████| 501/501 [01:39<00:00,  5.02it/s]


  Predictions saved: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_region_activity_regression/predict/predictions.csv  (2002 regions)
Finished stage 3

All stages completed!
Fine-tuned checkpoint: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_region_activity_regression/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=3-step=214.ckpt
Predictions: output_region_activity_regression/predict/predictions.csv


### 2.Interpretation

After building a model, we can use the interpretation layer of ChromBERT-tools to derive biological insights.

Here, we identify important factors between different region groups as an example.

In [3]:
region1_file = "./output_region_activity_regression/dataset/up.csv"  # Regions with increased activity generated by the command above
region2_file = "./output_region_activity_regression/dataset/nochange.csv"  # Unchanged regions generated by the command above

In [4]:
import glob
ft_ckpt_dir = "./output_region_activity_regression/train/**/*.ckpt"  # Use checkpoints generated by the command above

ft_ckpt = glob.glob(ft_ckpt_dir, recursive=True)[0]
ft_ckpt

'./output_region_activity_regression/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=3-step=214.ckpt'

In [9]:
# The ``interpret_regulator_effects_between_region_groups`` command generates regulator embeddings for different region groups and calculates the embedding difference of each regulator between groups as 1 − cosine similarity. Regulators with larger embedding differences are considered more likely to be key regulators.

# --region1_file: the region group 1
# --region2_file: the region group 2
# --odir: output directory
# --genome: genome
# --resolution: resolution
# --ft-ckpt: fine-tuned model checkpoint
# --model-config:  model configuration generated by the the command above
# --data-config:  data configuration generated by the the command above


# Set the GPU device with CUDA_VISIBLE_DEVICES
!export CUDA_VISIBLE_DEVICES=1
!chrombert-tools interpret_regulator_effects_between_region_groups \
    --region1-file "./output_region_activity_regression/dataset/up.csv" \
    --region2-file "./output_region_activity_regression/dataset/nochange.csv" \
    --odir "./output_region_activity_regression" \
    --genome "hg38" \
    --resolution "1kb" \
    --ft-ckpt {ft_ckpt} \
    --model-config "./output_region_activity_regression/model_config.json" \
    --data-config './output_region_activity_regression/dataset_config.json'
        

Region summary - total: 1000, overlapping with ChromBERT: 1000 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
Region summary - total: 1000, overlapping with ChromBERT: 1000 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 0
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/cli/output_region_activity_regression/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=3-step=214.ckpt
Loading from pl module, remove prefix 'model.'
Loading from pl module, replace 'pretrain_model' with 'pretrain_model.chrombert'
Loaded 111/111 parameters
100%|███████████████████████████

In [10]:
import pandas as pd
factor_importance_rank = pd.read_csv("output_region_activity_regression/results/factor_importance_rank.csv")
factor_importance_rank.head(n=25)

,factors,similarity,rank,embedding_shift
0,pax3-foxo1a,0.161664,1,0.838336
1,myog,0.172447,2,0.827553
2,myf5,0.220504,3,0.779496
3,pax7,0.320780,4,0.679220
4,myod1,0.350623,5,0.649377
5,tead1,0.372841,6,0.627159
6,neurod1,0.373016,7,0.626984
7,dux4,0.373749,8,0.626251
8,ascl1,0.395700,9,0.604300
9,neurog2,0.469495,10,0.530505
